In [ ]:
from _utils_siamese import *

from sklearn.utils import resample

In [ ]:
def load_train_split(train_file):
    train_split = pd.read_csv(train_file)
    return train_split
    
def load_test_split(test_file):
    test_split = pd.read_csv(test_file)
    return test_split

# Calculate performance metrics and 95% confidence intervals
def calculate_metrics_with_ci(predictions, labels, n_bootstrap=1000, seed=42):
    np.random.seed(seed)
    
    # --- Calculate point estimates for all samples ---
    point_metrics = {
        'ROC': roc_auc_score(labels, predictions),
        'Accuracy': accuracy_score(labels, predictions.round()),
        'Precision': precision_score(labels, predictions.round(), zero_division=0),
        'Recall': recall_score(labels, predictions.round()),
        'F1 Score': f1_score(labels, predictions.round())
    }
    
    # --- Bootstrap for confidence interval calculation ---
    boot_metrics = {
        'ROC': [],
        'Accuracy': [],
        'Precision': [],
        'Recall': [],
        'F1 Score': []
    }
    
    # Bootstrap resampling
    for _ in range(n_bootstrap):
        pred_sample, label_sample = resample(predictions, labels)
        # Calculate metrics for the current subset
        boot_metrics['ROC'].append(roc_auc_score(label_sample, pred_sample))
        boot_metrics['Accuracy'].append(accuracy_score(label_sample, pred_sample.round()))
        boot_metrics['Precision'].append(precision_score(label_sample, pred_sample.round(), zero_division=0))
        boot_metrics['Recall'].append(recall_score(label_sample, pred_sample.round()))
        boot_metrics['F1 Score'].append(f1_score(label_sample, pred_sample.round()))
    
    result = {}
    for metric in point_metrics.keys():
        result[metric] = {
            'Point': round(point_metrics[metric], 3),  
            'CI_Lower': round(np.percentile(boot_metrics[metric], 2.5), 3),  
            'CI_Upper': round(np.percentile(boot_metrics[metric], 97.5), 3)   
        }
    return result


def save_predictions(fold_dir, filename, predictions, labels):
    data = {'Predictions': predictions, 'Labels': labels}
    df_save = pd.DataFrame(data)
    df_save.to_csv(os.path.join(fold_dir, filename), index=False)


# Save performance metrics for each fold's training and testing
def save_metrics(fold_num, train_metrics, test_metrics, history_metrics, output_file):
    fold_metrics = {
        'Fold': fold_num,
        'Train_ROC': f"{train_metrics['ROC']['Point']}({train_metrics['ROC']['CI_Lower']}-{train_metrics['ROC']['CI_Upper']})",
        'Train_Accuracy': f"{train_metrics['Accuracy']['Point']}({train_metrics['Accuracy']['CI_Lower']}-{train_metrics['Accuracy']['CI_Upper']})",
        'Train_Precision': f"{train_metrics['Precision']['Point']}({train_metrics['Precision']['CI_Lower']}-{train_metrics['Precision']['CI_Upper']})",
        'Train_Recall': f"{train_metrics['Recall']['Point']}({train_metrics['Recall']['CI_Lower']}-{train_metrics['Recall']['CI_Upper']})",
        'Train_F1 Score': f"{train_metrics['F1 Score']['Point']}({train_metrics['F1 Score']['CI_Lower']}-{train_metrics['F1 Score']['CI_Upper']})",
        'Test_ROC': f"{test_metrics['ROC']['Point']}({test_metrics['ROC']['CI_Lower']}-{test_metrics['ROC']['CI_Upper']})",
        'Test_Accuracy': f"{test_metrics['Accuracy']['Point']}({test_metrics['Accuracy']['CI_Lower']}-{test_metrics['Accuracy']['CI_Upper']})",
        'Test_Precision': f"{test_metrics['Precision']['Point']}({test_metrics['Precision']['CI_Lower']}-{test_metrics['Precision']['CI_Upper']})",
        'Test_Recall': f"{test_metrics['Recall']['Point']}({test_metrics['Recall']['CI_Lower']}-{test_metrics['Recall']['CI_Upper']})",
        'Test_F1 Score': f"{test_metrics['F1 Score']['Point']}({test_metrics['F1 Score']['CI_Lower']}-{test_metrics['F1 Score']['CI_Upper']})"
    }
    history_metrics.append(fold_metrics)
    df_metrics = pd.DataFrame(history_metrics)
    df_metrics.to_csv(output_file, index=False)


In [ ]:
Class_num = 2
Size = (64, 64, 64) 
Batchsize = 8
dropout_rate = 0.5
weight_decay = 1e-4
Epoch_num = 50

adam = Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, decay=0.0)

params = {'dim': Size,
          'batch_size': Batchsize,
          'n_classes': Class_num,
          'n_channels': 1,
          'shuffle': True}

params_val = {'dim': Size,
              'batch_size': 1,
              'n_classes': Class_num,
              'n_channels': 1,
              'shuffle': False}

In [ ]:
model_file_current = ".../STRANet/"

model_path = ".../STRANet/Model/final_epoch_fold4_model.h5"

model = build_siamese_CNN(Size, weight_decay, dropout_rate, Class_num)

model.load_weights(model_path)

train_split = load_train_split(".../STRANet/Data/train_data.csv")
test_split = load_test_split(".../STRANet/Data/test_data.csv")

train_generator = DataGenerator(train_split[['Pre_path', 'Post_pseudo_path', 'Binary_Label']], **params_val) 
test_generator = DataGenerator(test_split[['Pre_path', 'Post_pseudo_path', 'Binary_Label']], **params_val)  

train_predictions, train_labels = get_predictions_and_labels(train_generator, model)
test_predictions, test_labels = get_predictions_and_labels(test_generator, model)

save_predictions(model_file_current, 'final_trian_predictions.csv', train_predictions, train_labels)
save_predictions(model_file_current, 'final_test_predictions.csv', test_predictions, test_labels)

history_metrics = []

train_metrics = calculate_metrics_with_ci(train_predictions, train_labels)
test_metrics = calculate_metrics_with_ci(test_predictions, test_labels)

fold_num = 4    # Manually change the fold number 

output_file = os.path.join(model_file_current, f'final_metrics_fold{fold_num}.csv')

save_metrics(fold_num=fold_num, 
             train_metrics=train_metrics, 
             test_metrics=test_metrics, 
             history_metrics=history_metrics, 
             output_file=output_file)